# 04 - Full Simulations & Analysis

So far, we have cleaned real ATP games data and got familiar with it (notebook 00). We then used Zermelo's algorithm to extract the strengths of players and get their true strengths for each year (notebook 01). Based on this truth, we calibrated the mathematical model to get parameters for creating synthetic data (maximum strength distribution, aging curves, retirement process, ...) (notebook 02). We ran an "empty" simulation (no tournaments) to validate the number of active players in the long term (stable number of active players, good potential distribution,...) (notebook 03). 

Now that we have everything we need for the tournaments simulation (with the functions contained in the `tournaments.py`), we can simulate the ATP circuit and compare the different rankings metrics (ATP points, Zermelo's strengths, PageRank scores, In-Degree Scores) with the hidden truth of our simulated data.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# importing the parameters stored in the .json file
import json

config_path = "../config/simulation_params.json"

with open(config_path, "r") as f:
    config_params = json.load(f)

In [4]:
from tournaments import run_full_tournaments

In [5]:
# for autoreload the modifications of the functions in the simulation.py file
%load_ext autoreload
%autoreload 2

## Get and Save the tournaments schedule and points

In [6]:
tournament_points_path = "../config/rules/tournament_properties.txt"
tournament_schedule_path = "../config/rules/tournament_schedule.txt"

tournaments_points_raw = pd.read_csv(tournament_points_path, comment="#", sep="\t")
tournaments_schedule = pd.read_csv(tournament_schedule_path, comment="#", sep=",", header=None, names=["date", "level"])

In [7]:
# get the week of each tournament
tournaments_schedule["week"] = pd.factorize(tournaments_schedule["date"])[0]

As we have the number of ITF "10" and "20" tournaments but not the calendar, I had to add them randomly over the whole year (47 weeks, from 0 to 46):

In [8]:
# add the small tournaments (10 and 20) into the schedule
number_ITF_10_tournaments = tournaments_points_raw.loc[tournaments_points_raw["level"] == 10, "number"].iloc[0]
number_ITF_20_tournaments = tournaments_points_raw.loc[tournaments_points_raw["level"] == 20, "number"].iloc[0]

print(f"Number of ITF 10 tournaments: {number_ITF_10_tournaments}")
print(f"Number of ITF 20 tournaments: {number_ITF_20_tournaments}")

# creating dataframes for the ITF 10 and 20 tournaments (attributing them one week each)
weeks_ITF_10 = np.random.randint(0, 47, size=number_ITF_10_tournaments)
weeks_ITF_20 = np.random.randint(0, 47, size=number_ITF_20_tournaments)

ITF_10_schedule = pd.DataFrame({"level": 10, "week": weeks_ITF_10})
ITF_20_schedule = pd.DataFrame({"level": 20, "week": weeks_ITF_20})

# get the full schedule
tournaments_schedule_full = pd.concat([ITF_10_schedule, ITF_20_schedule, tournaments_schedule])

# merge and keep only the relevant columns for the simulation
tournament_schedule_final = pd.merge(tournaments_schedule_full, tournaments_points_raw, how="left")
tournament_schedule_final.drop(columns=["date", "number", "W", "F", "SF", "QF", "R16", "R32", "R64", "R128", "Q", "Q1", "Q2", "Q3"], inplace=True)
tournament_schedule_final.sort_values(by=["week", "level"], ascending=[True, False], inplace=True)
tournament_schedule_final.reset_index(drop=True, inplace=True)

# create the tournament_points file with the points for each round and each tournament level
tournaments_points = tournaments_points_raw[["level", "W", "F", "SF", "QF", "R16", "R32", "R64", "R128", "Q", "Q1", "Q2", "Q3"]].copy()

Number of ITF 10 tournaments: 385
Number of ITF 20 tournaments: 155


In [9]:
tournament_schedule_final.to_csv("../data/processed/tournaments_schedule.csv", index=False)
tournaments_points.to_csv("../data/processed/tournament_points.csv", index=False)

In [10]:
games, rankings = run_full_tournaments(years=25, 
                         config_params=config_params,
                         tournaments_schedule=tournament_schedule_final,
                         tournaments_points=tournaments_points,
                         seeding=True)

Initialising the tournament by running a warm-up simulation...
End of the initialisation. Starting the simulation of the tournaments...


Simulating year 25/25...: 100%|██████████| 25/25 [00:53<00:00,  2.15s/it]
